In [0]:
-- ============================================================================
-- dbxWearables ZeroBus — Enable Predictive Optimization
-- ============================================================================
--
-- Checks whether predictive optimization is enabled on the target schema
-- and optionally enables it. Required for liquid clustering (CLUSTER BY AUTO)
-- on the wearables_zerobus bronze table.
--
-- Predictive optimization inheritance: account → catalog → schema
-- If the catalog already has it enabled, the schema inherits automatically.
--
-- Reference:
--   https://docs.databricks.com/en/optimizations/predictive-optimization/
-- ============================================================================

-- Bind widget parameters to session variables
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT :catalog_use;

DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);

-- Tag this session for cost tracking and auditing
SET QUERY_TAGS['project'] = :project_tag,
    QUERY_TAGS['businessUnit'] = :business_unit_tag,
    QUERY_TAGS['env'] = :env_tag,
    QUERY_TAGS['component'] = 'predictive_optimization',
    QUERY_TAGS['catalog'] = :catalog_use,
    QUERY_TAGS['schema'] = :schema_use;

-- Step 1: Check current predictive optimization status
-- Look for "Predictive Optimization" in the output — values:
--   EnabledByInheritance  → already active, no action needed
--   Enabled               → explicitly enabled on this schema
--   Disabled              → needs to be enabled (run Step 2)
DESCRIBE SCHEMA EXTENDED IDENTIFIER(catalog_use || '.' || schema_use);

-- Step 2: Enable predictive optimization
-- WARNING: This sets an explicit ENABLE on the schema, overriding
-- the inheritance chain. If the catalog already provides it, prefer
-- leaving inheritance intact.
EXECUTE IMMEDIATE
  'ALTER SCHEMA ' || catalog_use || '.' || schema_use || ' ENABLE PREDICTIVE OPTIMIZATION';

-- Step 3: Verify the change
DESCRIBE SCHEMA EXTENDED IDENTIFIER(catalog_use || '.' || schema_use);